# Customer Cohort & Retention Analysis
**Author:** Aaryaraj Patil  
**Concepts:** Acquisition Cohorts, Retention Heatmap, Churn Dynamics, E-Commerce Marketplace Retention Curves.

---

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data_dir = os.path.join('..', 'data', 'Brazilian E-Commerce Public Dataset by Olist')
orders = pd.read_csv(os.path.join(data_dir, 'olist_orders_dataset.csv'))
cust = pd.read_csv(os.path.join(data_dir, 'olist_customers_dataset.csv'))

orders = orders[orders['order_status'] == 'delivered'].copy()
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
df = pd.merge(orders[['order_id', 'customer_id', 'order_purchase_timestamp']],
              cust[['customer_id', 'customer_unique_id']],
              on='customer_id')

print('Delivered orders ready for cohort indexing:', len(df))

## 1. Computing Cohort Months & Period Indices

In [ ]:
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M')
df['cohort_month'] = df.groupby('customer_unique_id')['order_month'].transform('min')

df['cohort_index'] = (
    (df['order_month'].dt.year - df['cohort_month'].dt.year) * 12 +
    (df['order_month'].dt.month - df['cohort_month'].dt.month)
)

cohort_data = df.groupby(['cohort_month', 'cohort_index'])['customer_unique_id'].nunique().reset_index()
cohort_counts = cohort_data.pivot(index='cohort_month', columns='cohort_index', values='customer_unique_id').fillna(0)

cohort_sizes = cohort_counts.iloc[:, 0]
retention = cohort_counts.divide(cohort_sizes, axis=0) * 100
retention.iloc[:, :10].round(2)

## 2. Retention Matrix Interpretation & Churn Dynamics
- **Steep Drop-off:** Month 1 retention across all cohorts hovers between **0.2% and 0.7%**.
- **Business Model Implication:** Olist functions primarily as an acquisition-driven marketplace for durable/specialty goods rather than a high-frequency replenishment service (like grocery delivery).
- **Strategic Recommendation:** Implement post-purchase loyalty rewards, automated replenishment reminders, and personalized product bundles to convert one-off shoppers into repeat buyers.